# H8 사전 확인: Jev 판단기 정확도와 발췌를 본 재작성의 구제율

H8 다음 변형(1차 검색 상위 5개 발췌로 답할 수 있는지 판단하고, 못 하면 발췌를 보고 재작성) 전에 두 가지를 잰다. 판단기 정확도는 dev-small 1,039개에서 Jev 확률과 "기준(EXP-014) 상위 5개에 정답 없음"의 일치로, 구제율은 dev-full에서 기준이 놓친 177개 중 판단기가 고르고 재작성으로 정답을 상위 5개에 올린 비율로 잰다(사용자 결정 2026-10-15). 판단은 Jev(`configs/h8/judge_v1.yaml`), 재작성은 Gemma 4 31B 사고 과정 끔(`configs/rewriter/prompt/rwctx_v1.yaml`). 결과는 다음 변형의 근거로만 쓴다.

In [1]:
import json
import os
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from langsmith import traceable
from langsmith.run_trees import get_cached_client
from omegaconf import OmegaConf

from rag.conditions import build_extractor
from rag.eval.metrics import load_gold, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever, chunks_by_doc
from rag.rewrite import Rewriter, parse
from rag.search import embed_queries, query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
OUT = ROOT / 'data' / 'precheck'
PREFIX = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-k20'
TOP = 5
JUDGE = OmegaConf.load(ROOT / 'configs' / 'h8' / 'judge_v1.yaml')
OVERRIDES = ['+exp=exp014', '+rewriter=gemma_openrouter', 'rewriter/prompt=rwctx_v1',
             '+retriever.rewrite={enabled: false, lists: [dense, bm25], max_chars: 300}']

In [3]:
def setup(scope):
    """EXP-014 검색기와 질의(임베딩, BM25 토큰, 조건)."""
    cfg = load_cfg([*OVERRIDES, f'index.scope={scope}'])
    ret = Retriever(cfg)
    qids, path = query_list(cfg)
    texts = read_texts(path, qids)
    ex = build_extractor(cfg)
    return {'cfg': cfg, 'ret': ret, 'qids': qids, 'texts': dict(zip(qids, texts, strict=True)),
            'emb': dict(zip(qids, query_embeddings(cfg, qids, path, query_files(cfg)['emb']),
                            strict=True)),
            'tokens': dict(zip(qids, ret.query_tokens(texts), strict=True)),
            'conds': {q: ex.extract(t) for q, t in zip(qids, texts, strict=True)},
            'by_doc': chunks_by_doc(ret.dense_ids)}


def best_chunks(ctx, q, docs):
    """문서마다 질의와 dense 점수가 가장 높은 청크 번호(dense_ids 순서)."""
    row = ctx['ret'].chunk_emb @ ctx['emb'][q]
    return [int(ctx['by_doc'][d][np.argmax(row[ctx['by_doc'][d]])]) for d in docs]


def chunk_texts(dirs, wanted):
    """청크 번호(인덱스들을 이어 붙인 순서) → 본문. 필요한 줄만 읽는다."""
    out, base, wanted = {}, 0, set(wanted)
    for d in dirs:
        with (d / 'chunks.jsonl').open(encoding='utf-8') as f:
            for i, line in enumerate(f):
                if base + i in wanted:
                    out[base + i] = json.loads(line)['text']
        base += i + 1
    return out


def as_block(passages):
    return '\n\n'.join(f'[발췌 {i}]\n{p}' for i, p in enumerate(passages, 1))


def load_cache(path, basis):
    if not path.exists():
        return {}
    saved = json.loads(path.read_text(encoding='utf-8'))
    assert saved['basis'] == basis, (saved['basis'], basis)
    return saved['items']


def save_cache(path, basis, items):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({'basis': basis, 'items': items}, ensure_ascii=False),
                    encoding='utf-8')


@traceable(run_type='llm', name='h8_judge',
           process_inputs=lambda i: {'query': i['query'], 'passages': i['passages']})
def ask_judge(query, passages):
    body = {'model': JUDGE.model, 'state': {'질의': query, '검색 발췌': passages},
            'questions': {'q': {'type': 'noul', 'instructions': JUDGE.instructions,
                                'criteria': OmegaConf.to_container(JUDGE.criteria)}}}
    req = urllib.request.Request(JUDGE.url, data=json.dumps(body).encode(), headers={
        'Authorization': f'Bearer {os.environ[JUDGE.api_key_env]}',
        'Content-Type': 'application/json'})
    start = time.perf_counter()
    with urllib.request.urlopen(req, timeout=JUDGE.timeout) as r:
        resp = json.loads(r.read().decode())
    return {'p': resp['answers']['q']['noul'], 'usage': resp['usage'],
            'seconds': round(time.perf_counter() - start, 3)}


def judge_all(qids, queries, passages, path):
    """질의마다 답할 수 있을 확률. 캐시에 없는 것만 호출한다."""
    basis = {'model': JUDGE.model, 'version': JUDGE.version}
    items = load_cache(path, basis)
    todo = [q for q in qids if q not in items]
    try:
        with ThreadPoolExecutor(JUDGE.workers) as pool:
            def one(q):
                meta = {'qid': q, 'model': JUDGE.model, 'version': JUDGE.version}
                return ask_judge(queries[q], passages[q], langsmith_extra={'metadata': meta})

            for q, rec in zip(todo, pool.map(one, todo), strict=True):
                items[q] = rec
    finally:
        get_cached_client().flush()
        save_cache(path, basis, items)
    print(f'판단 {len(qids)}건(호출 {len(todo)}건)')
    return items


def rewrite_all(cfg, qids, queries, passages, path):
    """발췌를 보고 쓴 재작성 응답. 캐시에 없는 것만 호출한다."""
    w = cfg.rewriter
    basis = {'model': w.model, 'reasoning': w.reasoning, 'version': w.prompt.version}
    items = load_cache(path, basis)
    todo = [q for q in qids if q not in items]
    rw = Rewriter(cfg)
    try:
        with ThreadPoolExecutor(w.concurrency) as pool:
            for q, rec in zip(todo, pool.map(
                    lambda q: rw.call(queries[q], q, passages=as_block(passages[q])), todo),
                    strict=True):
                items[q] = rec
    finally:
        get_cached_client().flush()
        save_cache(path, basis, items)
    print(f'재작성 {len(qids)}건(호출 {len(todo)}건)')
    return items


def search_rewrites(ctx, rewrites):
    """재작성 질의만으로 검색한 상위 10개와, 원 질의 목록에 재작성 dense·BM25 목록을 더한 상위 10개.
    조건은 원 질의 것을 쓴다(EXP-013과 같음)."""
    ret, qs = ctx['ret'], list(rewrites)
    emb = embed_queries(ctx['cfg'], [rewrites[q] for q in qs])
    tok = ret.query_tokens([rewrites[q] for q in qs])
    only, added = {}, {}
    for q, e, t in zip(qs, emb, tok, strict=True):
        rw_row = ret.chunk_emb @ e
        only[q] = ret.rank(rw_row, t, ctx['conds'][q])
        added[q] = ret.explain(ret.chunk_emb @ ctx['emb'][q], ctx['tokens'][q], ctx['conds'][q],
                               rewrite=(rw_row, t))[0]
    return only, added

## dev-small: 1차 검색과 판단 입력

EXP-014 설정으로 상위 5개를 다시 만들고 순위 파일과 같은지 본다. 문서마다 최고점 청크를 발췌로 쓴다.

In [4]:
ctx = setup('dev-small')
ret = ctx['ret']
gold = load_gold(ctx['cfg'], 'dev-small')
top = {q: ret.rank(ret.chunk_emb @ ctx['emb'][q], ctx['tokens'][q], ctx['conds'][q])
       for q in ctx['qids']}
run = read_run(RUNS / f'{PREFIX}-dev-small.csv').sort_values(['qid', 'rank'])
run_top = run.groupby('qid')['doc_id'].apply(list)
print('순위 파일과 같은 질의', sum(top[q] == run_top[q] for q in ctx['qids']), '/', len(top))
miss = {q: gold.loc[q, 'doc_id'] not in top[q][:TOP] for q in ctx['qids']}
print('기준 상위 5개 실패', sum(miss.values()))

순위 파일과 같은 질의 1039 / 1039
기준 상위 5개 실패 13


In [5]:
picks = {q: best_chunks(ctx, q, top[q][:TOP]) for q in ctx['qids']}
texts = chunk_texts(ret.dirs, [i for v in picks.values() for i in v])
passages = {q: [texts[i] for i in v] for q, v in picks.items()}
lens = pd.Series([len(p) for v in passages.values() for p in v])
print('발췌 글자 수', lens.describe()[['mean', '50%', 'max']].round(0).to_dict())

발췌 글자 수 {'mean': 670.0, '50%': 537.0, 'max': 1152.0}


## 판단: 시험 5건

In [6]:
trial = ctx['qids'][:5]
judged = judge_all(trial, ctx['texts'], passages, OUT / 'h8_judge_dev-small.json')
for q in trial:
    print(q, round(judged[q]['p'], 3), '기준 실패' if miss[q] else '기준 성공', judged[q]['usage'])

판단 5건(호출 5건)
021857-0001 0.88 기준 성공 {'input_tokens': 2662, 'output_tokens': 20, 'cost': 0.000111804}
021890-0001 0.59 기준 성공 {'input_tokens': 3268, 'output_tokens': 20, 'cost': 0.000137256}
028740-0001 0.87 기준 성공 {'input_tokens': 3078, 'output_tokens': 20, 'cost': 0.000129276}
028866-0001 0.87 기준 성공 {'input_tokens': 2708, 'output_tokens': 20, 'cost': 0.000113736}
028893-0001 0.93 기준 성공 {'input_tokens': 2408, 'output_tokens': 20, 'cost': 0.000101136}


## 판단: dev-small 전체

In [7]:
judged = judge_all(ctx['qids'], ctx['texts'], passages, OUT / 'h8_judge_dev-small.json')
jd = pd.DataFrame({'p': {q: judged[q]['p'] for q in ctx['qids']}, 'miss': miss})
sec = pd.Series([judged[q]['seconds'] for q in ctx['qids']])
cost = sum(judged[q]['usage'].get('cost') or 0 for q in ctx['qids'])
print('확률 분포', jd['p'].describe()[['mean', '25%', '50%', '75%']].round(3).to_dict())
print('지연 p50·p95(초)', round(sec.median(), 2), round(sec.quantile(0.95), 2))
print('비용 $', round(cost, 4))

판단 1039건(호출 1034건)
확률 분포 {'mean': 0.777, '25%': 0.7, '50%': 0.83, '75%': 0.9}
지연 p50·p95(초) 0.41 0.82
비용 $ 0.1243


## 판단기 정확도

"불가"(확률 < 임계값)가 기준 실패를 얼마나 잡는지(재현율)와, 고른 질의 중 실제 실패 비율(정밀도). 재작성 대상 임계값은 0.5로 정했다(라벨을 보기 전). 다른 임계값은 곡선으로만 본다.

In [8]:
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pick = jd['p'] < t
    hit = int((pick & jd['miss']).sum())
    rows.append({'임계값': t, '고른 질의': int(pick.sum()), '그중 실패': hit,
                 '정밀도': hit / pick.sum() if pick.sum() else np.nan,
                 '재현율': hit / jd['miss'].sum()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('실패 질의 확률', jd.loc[jd['miss'], 'p'].round(3).sort_values().tolist())

 임계값  고른 질의  그중 실패   정밀도   재현율
 0.1      0      0   NaN 0.000
 0.2      1      0 0.000 0.000
 0.3     10      0 0.000 0.000
 0.4     25      0 0.000 0.000
 0.5     76      1 0.013 0.077
 0.6    151      2 0.013 0.154
 0.7    256      7 0.027 0.538
 0.8    430      9 0.021 0.692
 0.9    774     12 0.016 0.923
실패 질의 확률 [0.43, 0.52, 0.63, 0.63, 0.68, 0.68, 0.68, 0.7, 0.71, 0.83, 0.86, 0.87, 0.92]


In [3]:
# 판단기가 실패를 성공보다 낮게 매기는 정도(AUC). 확률은 캐시에서 다시 읽는다
saved = json.loads((OUT / 'h8_judge_dev-small.json').read_text(encoding='utf-8'))['items']
gold_s = load_gold(load_cfg(), 'dev-small')
run_s = read_run(RUNS / f'{PREFIX}-dev-small.csv').sort_values(['qid', 'rank'])
top_s = run_s.groupby('qid')['doc_id'].apply(list)
fail = pd.Series({q: gold_s.loc[q, 'doc_id'] not in top_s[q][:TOP] for q in gold_s.index})
p = pd.Series({q: saved[q]['p'] for q in gold_s.index})
pm, ph = p[fail].to_numpy(), p[~fail].to_numpy()
auc = ((pm[:, None] < ph[None, :]) + 0.5 * (pm[:, None] == ph[None, :])).mean()
print('실패', len(pm), '평균 p', round(pm.mean(), 3), ', 성공 평균 p', round(ph.mean(), 3))
print('AUC(실패를 낮게)', round(auc, 3))

실패 13 평균 p 0.703 , 성공 평균 p 0.778
AUC(실패를 낮게) 0.672


## dev-small: "불가" 질의 재작성

확률 0.5 미만 질의만 발췌를 보고 재작성한다. 재작성 질의만으로 검색한 경우와 원 질의 목록에 더한 경우(EXP-013 방식)를 함께 보고, 기준 실패를 살린 수와 기준 성공을 잃은 수를 센다.

In [9]:
target = [q for q in ctx['qids'] if jd.loc[q, 'p'] < JUDGE.rewrite_below]
raw = rewrite_all(ctx['cfg'], target, ctx['texts'], passages, OUT / 'h8_rw_dev-small.json')
rewrites = {q: parse(raw[q]['raw'], ctx['cfg'].retriever.rewrite.max_chars) for q in target}
print('대상', len(target), ', 형식 실패', sum(v is None for v in rewrites.values()))
rewrites = {q: v for q, v in rewrites.items() if v}

재작성 76건(호출 76건)
대상 76 , 형식 실패 0


In [10]:
def tally(ctx, gold_doc, base_top, rewrites):
    only, added = search_rewrites(ctx, rewrites)
    rows = []
    for name, res in [('재작성 질의만', only), ('원 질의 + 재작성 목록', added)]:
        was = {q: gold_doc[q] in base_top[q][:TOP] for q in rewrites}
        now = {q: gold_doc[q] in res[q][:TOP] for q in rewrites}
        rows.append({'방식': name, '대상': len(rewrites),
                     '기준 실패 중 살림': sum(now[q] and not was[q] for q in rewrites),
                     '기준 실패 수': sum(not was[q] for q in rewrites),
                     '기준 성공 중 잃음': sum(was[q] and not now[q] for q in rewrites)})
    return pd.DataFrame(rows)


small = tally(ctx, gold['doc_id'], top, rewrites)
print(small.to_string(index=False))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2133.90it/s]


           방식  대상  기준 실패 중 살림  기준 실패 수  기준 성공 중 잃음
      재작성 질의만  76           0        1           0
원 질의 + 재작성 목록  76           0        1           0


## dev-full: 기준 실패 177개

전체 인덱스로 실패 질의만 판단하고, 0.5 미만이면 재작성한다. 구제율은 177개 전체 대비와 판단기가 고른 질의 대비를 함께 본다.

In [11]:
del ctx, ret
full = setup('full')
fret = full['ret']
fgold = load_gold(full['cfg'], 'dev-full')
frun = read_run(RUNS / f'{PREFIX}-dev-full.csv').sort_values(['qid', 'rank'])
frun_top = frun.groupby('qid')['doc_id'].apply(list)
fmiss = [q for q in full['qids'] if fgold.loc[q, 'doc_id'] not in frun_top[q][:TOP]]
ftop = {q: fret.rank(fret.chunk_emb @ full['emb'][q], full['tokens'][q], full['conds'][q])
        for q in fmiss}
print('기준 실패', len(fmiss))
print('순위 파일과 같은 질의', sum(ftop[q] == frun_top[q] for q in fmiss))

기준 실패 177
순위 파일과 같은 질의 177


In [12]:
fpicks = {q: best_chunks(full, q, ftop[q][:TOP]) for q in fmiss}
ftexts = chunk_texts(fret.dirs, [i for v in fpicks.values() for i in v])
fpass = {q: [ftexts[i] for i in v] for q, v in fpicks.items()}
fjudged = judge_all(fmiss, full['texts'], fpass, OUT / 'h8_judge_dev-full-miss.json')
fp = pd.Series({q: fjudged[q]['p'] for q in fmiss})
ftarget = list(fp.index[fp < JUDGE.rewrite_below])
print('판단기가 고른 실패', len(ftarget), '/', len(fmiss))

판단 177건(호출 177건)
판단기가 고른 실패 14 / 177


In [13]:
fraw = rewrite_all(full['cfg'], ftarget, full['texts'], fpass, OUT / 'h8_rw_dev-full-miss.json')
frw = {q: parse(fraw[q]['raw'], full['cfg'].retriever.rewrite.max_chars) for q in ftarget}
print('형식 실패', sum(v is None for v in frw.values()))
frw = {q: v for q, v in frw.items() if v}
fres = tally(full, fgold['doc_id'], ftop, frw)
fres['177개 대비 구제율'] = (fres['기준 실패 중 살림'] / len(fmiss)).round(3)
print(fres.to_string(index=False))

재작성 14건(호출 14건)
형식 실패 0


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1738.25it/s]


           방식  대상  기준 실패 중 살림  기준 실패 수  기준 성공 중 잃음  177개 대비 구제율
      재작성 질의만  14           1       14           0        0.006
원 질의 + 재작성 목록  14           1       14           0        0.006


## 호출 수·비용·지연

In [14]:
calls = {'Jev dev-small': list(judged.values()), 'Jev dev-full 실패': list(fjudged.values()),
         'Gemma dev-small': list(raw.values()), 'Gemma dev-full 실패': list(fraw.values())}
summary = []
for name, recs in calls.items():
    s = pd.Series([r['seconds'] for r in recs])
    summary.append({'항목': name, '호출': len(recs),
                    '비용($)': round(sum((r['usage'] or {}).get('cost') or 0 for r in recs), 4),
                    'p50(초)': round(s.median(), 2), 'p95(초)': round(s.quantile(0.95), 2)})
print(pd.DataFrame(summary).to_string(index=False))

               항목   호출  비용($)  p50(초)  p95(초)
    Jev dev-small 1039 0.1243    0.41    0.82
  Jev dev-full 실패  177 0.0198    0.49    2.00
  Gemma dev-small   76 0.0184    1.29    2.81
Gemma dev-full 실패   14 0.0029    1.26    5.14
